# Estadística de anillos en boro-grafidiino (BGDY): tutorial

Este cuaderno reproduce, paso a paso, el análisis de **estadística de anillos** del artículo:

> H. T. T. Nguyen, *Graphene layer of hybrid graphene/hexagonal boron nitride model upon heating*, Carbon Letters **29**, 521–528 (2019). https://doi.org/10.1007/s42823-019-00056-6

y lo aplica a una capa de **BGDY** (átomos de B unidos por cadenas de butadiino, –C≡C–C≡C–).

## Qué hace exactamente el artículo

| Aspecto | Lo que dice el artículo |
|---|---|
| Configuraciones | Instantáneas de dinámica molecular (LAMMPS, potencial de Tersoff) a distintas temperaturas |
| Herramienta | Programa **ISAACS** (Le Roux y Petkov, *J. Appl. Cryst.* 43, 181, 2010) |
| Definición de anillo | Regla del **"camino más corto"** (*shortest path*) |
| Resultado | Fig. 6: **fracción de anillos frente a su tamaño** (número de átomos del anillo) a 3600 K y 4500 K |
| Complemento | Fig. 5: distribución del **número de coordinación**, calculada con el mismo criterio de enlace |
| No especifica | El **radio de corte** para definir enlaces, ni cuál de las variantes del camino más corto (King o Guttman) usa |

Como el artículo no da esos dos detalles, aquí los hacemos explícitos y vemos cómo afectan al resultado.

## Índice
1. Teoría: de la estructura al grafo y del grafo a los anillos  
2. Construir BGDY (y grafeno para validar)  
3. Elegir el radio de corte  
4. Grafo de enlaces y número de coordinación  
5. Implementación del algoritmo  
6. Validación con grafeno  
7. BGDY prístino  
8. Ejemplo: vacante de boro (los criterios no siempre coinciden)  
9. Ejemplo: desorden "térmico" (figura tipo Fig. 6)  
10. Ejemplo: BGDY decorado con un átomo de Ni  
11. Cómo aplicarlo a una trayectoria de MD real  
12. Lo mismo con `matscipy.rings.ring_statistics`  
13. Resumen y lista de comprobación

In [ ]:
# Si falta ASE:  %pip install ase
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from collections import Counter, defaultdict
from ase import Atoms, Atom
from ase.neighborlist import neighbor_list

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. Teoría

### 1.1 De la estructura a un grafo
La red atómica se convierte en un **grafo**: cada átomo es un nodo y dos átomos están enlazados si su distancia es menor que un **radio de corte** $r_c$:

$$(i,j)\in E \iff |\mathbf r_i-\mathbf r_j + \mathbf S\cdot\mathbf h| < r_c ,$$

donde $\mathbf h$ es la matriz de la celda y $\mathbf S$ el vector entero de imagen periódica. Habitualmente $r_c$ se toma en el **primer mínimo de la función de distribución radial** g(r).

### 1.2 ¿Qué es un "anillo"?
Un anillo es un camino cerrado en el grafo. Contar *todos* los ciclos no tiene sentido (hay combinatoriamente muchos), así que se usan criterios que se quedan solo con los "más cortos". ISAACS (y su programa hermano R.I.N.G.S., Le Roux y Jund, *Comput. Mater. Sci.* 49, 70, 2010) implementan varios:

* **Criterio de King** (King, *Nature* 213, 1112, 1967): para cada átomo *i* y cada par de vecinos *j*, *k*, el anillo es *i* más el camino más corto de *j* a *k* que **no pasa por *i***.
* **Criterio de Guttman** (Guttman, *J. Non-Cryst. Solids* 116, 145, 1990): para cada **enlace** *i–j*, el anillo es el camino más corto de *i* a *j* que **no usa ese enlace**, cerrado con el propio enlace.
* **Anillos SP estrictos o primitivos** (Franzblau, *Phys. Rev. B* 44, 4925, 1991): un ciclo cuenta solo si, para cualquier par de sus átomos, el camino más corto en la red va *por el propio anillo* (no hay "atajos").

La "regla del camino más corto" que cita el artículo corresponde a **King o Guttman**. En redes perfectas los tres criterios coinciden; en redes con defectos, no (lo veremos en la sección 8).

### 1.3 Condiciones periódicas
En una celda periódica, un camino puede salir por un lado y entrar por el otro. Para no confundir un anillo real con un camino que "da la vuelta" a la celda, cada nodo de la búsqueda es un par **(átomo, vector de imagen acumulado)**. Un ciclo solo se cierra si vuelve al **mismo átomo con el mismo vector de imagen**.

### 1.4 Cómo se presenta el resultado
Si $N(n)$ es el número de anillos distintos de tamaño $n$:

* **Fracción** (como en la Fig. 6 del artículo): $f(n) = N(n)/\sum_{n'} N(n')$
* **Anillos por celda unidad**: $R_C(n) = N(n)/N_\text{celdas}$

## 2. Construir las estructuras

**BGDY**: red tipo panal en la que los vértices son átomos de B (sp²) y las aristas son cadenas de butadiino. La celda unidad tiene 14 átomos (B₂C₁₂). Usamos las distancias PBE de Mortazavi *et al.* (*J. Mater. Chem. A* 6, 11022, 2018):  
d(B–C) = 1.512 Å, d(C≡C) = 1.235 Å, d(C–C) = 1.345 Å → constante de red ≈ 11.85 Å.

**Grafeno**: lo usamos como caso de control, porque su resultado se conoce de antemano (solo hexágonos).

In [ ]:
def build_graphene(a=2.46, vacuum=20.0):
    cell = np.array([[a, 0, 0], [a/2, a*np.sqrt(3)/2, 0], [0, 0, vacuum]])
    return Atoms('C2', scaled_positions=[[1/3, 1/3, 0.5], [2/3, 2/3, 0.5]],
                 cell=cell, pbc=(True, True, False))

def build_bgdy(d_BC=1.512, d_triple=1.235, d_single=1.345, vacuum=20.0):
    # Distancia B-B a lo largo de una cadena B-C≡C-C≡C-B
    d_BB = 2*d_BC + 2*d_triple + d_single
    a = np.sqrt(3)*d_BB                      # constante de la red hexagonal
    cell = np.array([[a, 0, 0], [a/2, a*np.sqrt(3)/2, 0], [0, 0, vacuum]])
    rB1 = np.array([1/3, 1/3, 0.5]) @ cell
    rB2 = np.array([2/3, 2/3, 0.5]) @ cell
    symbols, positions = ['B', 'B'], [rB1, rB2]
    steps = np.cumsum([d_BC, d_triple, d_single, d_triple])   # posiciones de los 4 C de cada cadena
    for f in ([2/3, 2/3, 0.5], [2/3, -1/3, 0.5], [-1/3, 2/3, 0.5]):   # los 3 B vecinos de B1
        rn = np.array(f) @ cell
        u = (rn - rB1)/np.linalg.norm(rn - rB1)
        for s in steps:
            symbols.append('C'); positions.append(rB1 + s*u)
    atoms = Atoms(symbols, positions=positions, cell=cell, pbc=(True, True, False))
    atoms.wrap()
    return atoms

bgdy = build_bgdy()
print('Fórmula:', bgdy.get_chemical_formula(), ' | átomos/celda:', len(bgdy))
print('Constante de red a = %.3f Å  (literatura PBE: 11.85 Å)' % bgdy.cell.cellpar()[0])

Funciones de dibujo: enlaces, átomos coloreados por elemento y anillos rellenos con un color según su tamaño.

In [ ]:
ELEM_COLOR = {'C': '#5F5E5A', 'B': '#3B9E5A', 'Ni': '#2F6DB5'}
_palette = plt.get_cmap('tab20').colors

def ring_color(n):
    fixed = {3: '#E24B4A', 4: '#EF9F27', 6: '#9FE1CB', 30: '#B5D4F4', 50: '#CECBF6', 60: '#F4C0D1'}
    return fixed.get(n, _palette[n % 20])

def ring_xy(atoms, ring):
    # Coordenadas cartesianas de un anillo dado como lista de (átomo, imagen),
    # trasladado para que su centro caiga dentro de la celda (solo para dibujar)
    xyz = np.array([atoms.positions[a] + np.array(s) @ atoms.cell for a, s in ring])
    frac = np.linalg.solve(atoms.cell.T, xyz.mean(0))
    shift = np.floor(frac[:2])
    xyz -= shift[0]*atoms.cell[0] + shift[1]*atoms.cell[1]
    return xyz[:, :2]

def plot_structure(atoms, cutoff, rings=(), ax=None, title='', show_cell=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    seen = set()
    for r in rings:
        n = len(r)
        ax.add_patch(Polygon(ring_xy(atoms, r), closed=True, fc=ring_color(n), ec='none', alpha=0.55,
                             label=f'{n}-anillo' if n not in seen else None))
        seen.add(n)
    I, J, D = neighbor_list('ijD', atoms, cutoff)
    for i, j, d in zip(I, J, D):
        if i < j or (i == j):
            p = atoms.positions[i]
            ax.plot([p[0], p[0]+d[0]], [p[1], p[1]+d[1]], color='#888780', lw=1, zorder=2)
    for sym in sorted(set(atoms.get_chemical_symbols())):
        m = np.array(atoms.get_chemical_symbols()) == sym
        ax.scatter(*atoms.positions[m, :2].T, s=12 if sym == 'C' else 30,
                   c=ELEM_COLOR.get(sym, 'k'), zorder=3, label=sym)
    if show_cell:
        c = atoms.cell
        corners = np.array([[0, 0, 0], c[0], c[0]+c[1], c[1], [0, 0, 0]])
        ax.plot(corners[:, 0], corners[:, 1], 'k--', lw=0.6)
    ax.set_aspect('equal'); ax.set_title(title); ax.axis('off')
    ax.legend(loc='upper left', fontsize=8, frameon=False, bbox_to_anchor=(1, 1))
    return ax

plot_structure(bgdy.repeat((2, 2, 1)), 2.0, title='BGDY, supercelda 2×2 (aún sin anillos)');

## 3. Elegir el radio de corte

El artículo no da el valor de $r_c$. La forma estándar de elegirlo es mirar las distancias entre pares de átomos (o la g(r)) y situar $r_c$ **en el hueco entre la primera y la segunda capa de vecinos**.

En BGDY la primera capa está entre 1.24 y 1.51 Å (C≡C, C–C y B–C) y la segunda empieza en 2.58 Å. La distancia de 2.62 Å corresponde a los dos C unidos al mismo B. Cualquier $r_c$ entre ~1.6 y ~2.5 Å da el mismo grafo para la estructura perfecta. Tomamos **$r_c$ = 2.0 Å**, cerca del centro del hueco, para tener margen frente a las vibraciones.

In [ ]:
sc = bgdy.repeat((4, 4, 1))
rng = np.random.default_rng(0)
noisy = sc.copy(); noisy.positions += rng.normal(0, 0.08, noisy.positions.shape)

fig, ax = plt.subplots(figsize=(7, 3))
for at, lab, col in [(sc, 'BGDY perfecto', '#185FA5'), (noisy, 'BGDY con vibraciones (σ = 0.08 Å)', '#D85A30')]:
    d = neighbor_list('d', at, 3.2)
    ax.hist(d, bins=np.arange(1.0, 3.2, 0.02), histtype='step', lw=1.5, label=lab, color=col, density=True)
ax.axvline(2.0, color='k', ls='--', lw=1); ax.text(2.02, ax.get_ylim()[1]*0.8, '$r_c$ = 2.0 Å')
ax.set_xlabel('Distancia entre pares (Å)'); ax.set_ylabel('Densidad'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

print('Distancias únicas < 3 Å en el cristal perfecto:', np.unique(np.round(neighbor_list('d', bgdy, 3.0), 3)))

## 4. Grafo de enlaces y número de coordinación

`neighbor_list('ijS', ...)` de ASE devuelve, para cada enlace, los índices *i*, *j* y el vector de imagen **S**. Con eso construimos una lista de adyacencia. El número de coordinación Z (el de la Fig. 5 del artículo) es simplemente el número de vecinos de cada átomo.

El radio de corte puede ser un número (el mismo para todos los pares) o un diccionario por pares de elementos, p. ej. `{('B','C'): 2.0, ('C','C'): 2.0}`.

In [ ]:
ZERO = (0, 0, 0)

def build_graph(atoms, cutoff):
    # adj[i] = lista de (j, S): j es vecino de i a través de la imagen S
    I, J, S = neighbor_list('ijS', atoms, cutoff)
    adj = [[] for _ in range(len(atoms))]
    for a, b, s in zip(I, J, S):
        adj[a].append((int(b), (int(s[0]), int(s[1]), int(s[2]))))
    return adj

def coordination(adj):
    return dict(sorted(Counter(len(x) for x in adj).items()))

adj = build_graph(sc, 2.0)
print('Número de coordinación Z en BGDY 4×4:', coordination(adj))
print('→ los B tienen Z = 3 y los C de las cadenas Z = 2, como se espera')

## 5. Implementación del algoritmo

La pieza central es una **búsqueda en anchura (BFS)** sobre estados *(átomo, imagen)* que devuelve **todos** los caminos más cortos entre dos estados, con la opción de prohibir un nodo (criterio de King) o un enlace (criterio de Guttman).

Cada anillo se guarda en forma canónica (conjunto de estados trasladado a una referencia común). Así, el mismo anillo encontrado desde distintos átomos o enlaces, o en distintas imágenes periódicas, se cuenta **una sola vez**.

In [ ]:
def _add(s, S):
    return (s[0]+S[0], s[1]+S[1], s[2]+S[2])

def shortest_paths(adj, start, target, max_len, banned_states=(), banned_edge=None, max_paths=50):
    # Todos los caminos más cortos start -> target (estados = (átomo, imagen)), de longitud <= max_len
    banned_states = set(banned_states)
    dist, parents, frontier, depth = {start: 0}, defaultdict(list), [start], 0
    while frontier and depth < max_len and target not in dist:
        depth += 1
        nxt = []
        for st in frontier:
            a, s = st
            for b, S in adj[a]:
                ns = (b, _add(s, S))
                if ns in banned_states or (st, ns) == banned_edge:
                    continue
                if ns not in dist:
                    dist[ns] = depth; parents[ns].append(st); nxt.append(ns)
                elif dist[ns] == depth:
                    parents[ns].append(st)      # camino alternativo de igual longitud
        frontier = nxt
    if target not in dist:
        return []
    paths, stack = [], [(target, [target])]
    while stack and len(paths) < max_paths:   # reconstrucción hacia atrás
        st, acc = stack.pop()
        if st == start:
            paths.append(acc[::-1]); continue
        for p in parents[st]:
            stack.append((p, acc + [p]))
    return paths

def canonical(ring):
    # Clave única de un anillo, independiente del punto de partida, el sentido y la imagen periódica
    m = min(a for a, _ in ring)
    keys = [tuple(sorted((b, (t[0]-s[0], t[1]-s[1], t[2]-s[2])) for b, t in ring))
            for a, s in ring if a == m]
    return min(keys)

def find_rings(adj, max_size=40, criterion='guttman'):
    rings = {}
    for i in range(len(adj)):
        if criterion == 'guttman':                      # un anillo (o varios igual de cortos) por enlace
            for j, S in adj[i]:
                if j < i or (j == i and S < ZERO):
                    continue
                start, target = (i, ZERO), (j, S)
                for p in shortest_paths(adj, start, target, max_size-1, banned_edge=(start, target)):
                    rings.setdefault(canonical(p), p)
        elif criterion == 'king':                       # un anillo por átomo y par de vecinos
            nb = adj[i]
            for p_ in range(len(nb)):
                for q_ in range(p_+1, len(nb)):
                    for p in shortest_paths(adj, nb[p_], nb[q_], max_size-2, banned_states=[(i, ZERO)]):
                        ring = [(i, ZERO)] + p
                        rings.setdefault(canonical(ring), ring)
        else:
            raise ValueError("criterion: 'guttman' o 'king'")
    return list(rings.values())

def ring_statistics(atoms, cutoff, max_size=40, criterion='guttman'):
    adj = build_graph(atoms, cutoff)
    rings = find_rings(adj, max_size, criterion)
    counts = dict(sorted(Counter(len(r) for r in rings).items()))
    return {'rings': rings, 'counts': counts, 'coordination': coordination(adj), 'adj': adj}

def fractions(counts):
    tot = sum(counts.values())
    return {n: c/tot for n, c in counts.items()} if tot else {}

**Opcional: anillos SP estrictos (primitivos).** Se generan candidatos a partir de pares de caminos más cortos que salen de cada átomo y se descartan los que tienen un "atajo". Lo usaremos para comparar en la sección 8.

In [ ]:
def _bfs_dist(adj, start, max_depth):
    dist, frontier = {start: 0}, [start]
    for d in range(1, max_depth+1):
        nxt = []
        for a, s in frontier:
            for b, S in adj[a]:
                ns = (b, _add(s, S))
                if ns not in dist:
                    dist[ns] = d; nxt.append(ns)
        frontier = nxt
    return dist

def is_sp_ring(adj, ring):
    # True si ningún par de átomos del anillo está más cerca por fuera del anillo que por el anillo
    L = len(ring)
    for p in range(L):
        a, s = ring[p]
        dist = _bfs_dist(adj, (a, ZERO), L//2)
        for q in range(p+1, L):
            b, t = ring[q]
            if dist.get((b, (t[0]-s[0], t[1]-s[1], t[2]-s[2])), L) < min(q-p, L-(q-p)):
                return False
    return True

def _paths_to_root(parents, node, root, cap=5):
    out, stack = [], [(node, [node])]
    while stack and len(out) < cap:
        st, acc = stack.pop()
        if st == root:
            out.append(acc[::-1]); continue
        for p in parents[st]:
            stack.append((p, acc+[p]))
    return out

def find_sp_rings(adj, max_size=40):
    rings, half = {}, max_size//2
    for i in range(len(adj)):
        root = (i, ZERO)
        dist, parents, layers = {root: 0}, defaultdict(list), [[root]]
        for d in range(1, half+1):
            nxt = []
            for st in layers[-1]:
                a, s = st
                for b, S in adj[a]:
                    ns = (b, _add(s, S))
                    if ns not in dist:
                        dist[ns] = d; parents[ns].append(st); nxt.append(ns)
                    elif dist[ns] == d:
                        parents[ns].append(st)
            layers.append(nxt)
        cands = []
        for d in range(1, half+1):
            for v in layers[d]:
                ps = parents[v]
                if len(ps) >= 2 and 2*d <= max_size:                 # anillos pares
                    for x in range(len(ps)):
                        for y in range(x+1, len(ps)):
                            for P in _paths_to_root(parents, ps[x], root):
                                for Q in _paths_to_root(parents, ps[y], root):
                                    cands.append(P + [v] + Q[::-1][:-1])
                if 2*d+1 <= max_size:                                  # anillos impares
                    a, s = v
                    for b, S in adj[a]:
                        w = (b, _add(s, S))
                        if dist.get(w) == d and w > v:
                            for P in _paths_to_root(parents, v, root):
                                for Q in _paths_to_root(parents, w, root):
                                    cands.append(P + Q[::-1][:-1])
        for c in cands:
            if len(set(c)) == len(c):
                k = canonical(c)
                if k not in rings and is_sp_ring(adj, c):
                    rings[k] = c
    return list(rings.values())

## 6. Validación con grafeno

En grafeno perfecto todos los anillos son hexágonos y hay **un hexágono por cada 2 átomos**. Para una supercelda 4×4 (32 átomos) esperamos 16 anillos de tamaño 6.

In [ ]:
gr = build_graphene().repeat((4, 4, 1))
for crit in ['guttman', 'king']:
    res = ring_statistics(gr, cutoff=1.7, max_size=12, criterion=crit)
    print(f'{crit:8s}: {res["counts"]}   (átomos = {len(gr)})')
print('SP      :', dict(Counter(len(r) for r in find_sp_rings(build_graph(gr, 1.7), 12))))
plot_structure(gr, 1.7, ring_statistics(gr, 1.7, 12)['rings'], title='Grafeno 4×4: 16 hexágonos');

## 7. BGDY prístino

Cada poro de BGDY es un "hexágono gigante": 6 átomos de B y 6 cadenas de 4 C, es decir, un anillo de **30 átomos**. Como en el panal hay un poro por cada 2 vértices, esperamos **1 anillo de 30 por celda unidad**.

`max_size` tiene que ser mayor que el anillo más grande que queramos detectar: aquí, al menos 30.

In [ ]:
res = ring_statistics(sc, cutoff=2.0, max_size=40, criterion='guttman')
ncells = len(sc)//len(bgdy)
print('Anillos encontrados:', res['counts'])
print('Celdas unidad:', ncells, '→ R_C(30) =', res['counts'][30]/ncells, 'anillos/celda')
print('Fracción f(n):', fractions(res['counts']))
plot_structure(sc, 2.0, res['rings'], title='BGDY 4×4: 16 anillos de 30 átomos');

## 8. Ejemplo: vacante de boro (los criterios no siempre coinciden)

Quitamos un átomo de B. Las tres cadenas que salían de él quedan colgando (sus C extremos pasan a tener Z = 1), y los tres poros que rodeaban al B se fusionan en uno solo. Ese poro fusionado tiene 12 vértices de B y 12 cadenas, es decir, **60 átomos**.

Veamos qué detecta cada criterio:

In [ ]:
def central_index(atoms, symbol):
    c = atoms.cell.sum(0)/2
    idx = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s == symbol]
    return min(idx, key=lambda i: np.linalg.norm(atoms.positions[i, :2] - c[:2]))

vac = sc.copy()
del vac[central_index(vac, 'B')]

results = {}
for crit in ['guttman', 'king']:
    results[crit] = ring_statistics(vac, 2.0, max_size=80, criterion=crit)
    print(f'{crit:8s}:', results[crit]['counts'])
sp_rings = find_sp_rings(build_graph(vac, 2.0), 80)
print('SP      :', dict(sorted(Counter(len(r) for r in sp_rings).items())))
print('Coordinación:', results['guttman']['coordination'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_structure(vac, 2.0, results['guttman']['rings'], axes[0], 'Guttman')
plot_structure(vac, 2.0, [r for r in results['king']['rings'] if len(r) == 50][:1] +
               [r for r in results['king']['rings'] if len(r) == 30], axes[1], 'King (uno de los 50-anillos)')
plot_structure(vac, 2.0, sp_rings, axes[2], 'SP estrictos')
plt.tight_layout(); plt.show()

**Interpretación**

* **Guttman** no ve el poro fusionado. Cada enlace de su borde pertenece también a un poro intacto de 30, que es más corto, así que el anillo de 60 nunca es "el más corto" de ningún enlace.
* **King** tampoco lo ve, pero en su lugar encuentra anillos de **50**: el contorno de *dos poros intactos vecinos* (30 + 30 − 10: los 4 C de la cadena compartida desaparecen de ambos y sus 2 B se cuentan una sola vez). No son anillos "reales", porque la cadena que comparten es un atajo.
* **SP estrictos** dan el resultado intuitivo: 13 poros de 30 y 1 poro de 60.

👉 En redes perfectas da igual qué criterio uses. En redes con defectos o en estructuras fundidas (como las de 4500 K del artículo) **sí importa**, y hay que indicarlo al publicar.

## 9. Ejemplo: desorden "térmico" (figura tipo Fig. 6)

El artículo calcula la estadística sobre instantáneas de MD a alta temperatura. Aquí **no** hacemos MD: imitamos el efecto de las vibraciones con desplazamientos gaussianos aleatorios de amplitud σ (en cada componente). Es un modelo de juguete, pero muestra el mecanismo:

* Algunos enlaces se estiran por encima de $r_c$ y se "rompen": los poros se fusionan y aparecen anillos de 50, 70…
* Algunos átomos se acercan por debajo de $r_c$ (sobre todo los dos C unidos al mismo B, separados solo 2.62 Å): aparecen triángulos B–C–C y anillos de 29.

Promediamos sobre varias configuraciones por cada σ, como se haría con varias instantáneas de una trayectoria.

In [ ]:
def noisy_stats(base, sigma, cutoff, nframes=5, max_size=80, seed=0):
    rng = np.random.default_rng(seed)
    tot, coord = Counter(), Counter()
    for _ in range(nframes):
        x = base.copy(); x.positions += rng.normal(0, sigma, x.positions.shape)
        r = ring_statistics(x, cutoff, max_size, 'guttman')
        tot.update(r['counts']); coord.update(r['coordination'])
    return dict(sorted(tot.items())), dict(sorted(coord.items()))

sigmas = [0.05, 0.10, 0.15, 0.20]
data = {s: noisy_stats(sc, s, cutoff=2.0) for s in sigmas}

fig, axes = plt.subplots(2, len(sigmas), figsize=(15, 6), sharey='row')
for k, s in enumerate(sigmas):
    counts, coord = data[s]
    f = fractions(counts)
    axes[0, k].bar(list(f.keys()), list(f.values()), color='#D85A30', width=0.8)
    axes[0, k].set_title(f'σ = {s:.2f} Å'); axes[0, k].set_xlabel('Tamaño del anillo')
    axes[0, k].set_xlim(0, 82)
    tz = sum(coord.values())
    axes[1, k].bar(list(coord.keys()), [v/tz for v in coord.values()], color='#185FA5')
    axes[1, k].set_xlabel('Número de coordinación Z'); axes[1, k].set_xticks(range(0, 5))
axes[0, 0].set_ylabel('Fracción de anillos'); axes[1, 0].set_ylabel('Fracción de átomos')
plt.tight_layout(); plt.show()

for s in sigmas:
    print(f'σ = {s:.2f} Å  anillos: {data[s][0]}')

**Sensibilidad al radio de corte.** El mismo desorden da estadísticas distintas según $r_c$. Por eso, al comparar temperaturas o materiales, $r_c$ debe ser fijo y conviene indicarlo.

In [ ]:
print(f'{"r_c (Å)":>8} | ' + ' | '.join(f'σ={s:.2f}' for s in sigmas))
for rc in [1.85, 2.0, 2.2]:
    row = []
    for s in sigmas:
        counts, _ = noisy_stats(sc, s, cutoff=rc, nframes=3, seed=1)
        row.append(f'{counts.get(30, 0)/(3*ncells):6.2f}')
    print(f'{rc:8.2f} | ' + ' | '.join(row))
print('\nValores: anillos de 30 por celda (1.00 = estructura perfecta)')

Con $r_c$ = 1.85 Å los enlaces B–C (1.51 Å) se "rompen" enseguida. Con $r_c$ = 2.2 Å aparecen antes los triángulos B–C–C (los dos C unidos al mismo B están a 2.62 Å). El valor intermedio, 2.0 Å, en el centro del hueco entre capas de vecinos, es el más robusto.

## 10. Ejemplo: BGDY decorado con un átomo de Ni

Muy relevante si se estudian metales de transición sobre BGDY. Colocamos un Ni en la esquina de un poro, junto a un B y enlazado a los dos C α de dos cadenas distintas (d(Ni–C) ≈ 1.90 Å). Es una **posición ilustrativa, no optimizada**.

Con átomos de distinta especie conviene usar **radios de corte por pares**. La decisión de si el enlace Ni–B "cuenta" cambia la topología.

In [ ]:
base = bgdy.repeat((3, 3, 1))
kB = central_index(base, 'B')
I, J, D = neighbor_list('ijD', base, 2.0)
v1, v2 = D[I == kB][:2]                                # vectores B→C de dos cadenas
mid = (v1 + v2)/2; u = mid/np.linalg.norm(mid)
half = np.linalg.norm(v1 - v2)/2
s = np.linalg.norm(mid) + np.sqrt(1.90**2 - half**2)  # distancia B–Ni para que d(Ni–C) = 1.90 Å
ni = base.copy(); ni.append(Atom('Ni', base.positions[kB] + s*u))
print(f'd(Ni–B) = {s:.2f} Å')

cut_A = {('B', 'C'): 2.0, ('C', 'C'): 2.0, ('Ni', 'C'): 2.1}                # sin enlace Ni–B
cut_B = {**cut_A, ('Ni', 'B'): 2.3}                                          # con enlace Ni–B
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (lab, cut) in zip(axes, [('Ni–C solamente', cut_A), ('Ni–C y Ni–B', cut_B)]):
    r = ring_statistics(ni, cut, 40, 'guttman')
    print(f'{lab:16s} anillos: {r["counts"]}   Z: {r["coordination"]}')
    small = [x for x in r['rings'] if len(x) < 10]
    plot_structure(ni, cut, [x for x in r['rings'] if len(x) >= 10] + small, ax, lab)   # anillos pequeños encima
    ax.set_xlim(ni.positions[kB, 0]-10, ni.positions[kB, 0]+10); ax.set_ylim(ni.positions[kB, 1]-8, ni.positions[kB, 1]+9)
plt.tight_layout(); plt.show()

* **Solo Ni–C**: aparece un anillo de 4 (Ni–C–B–C). El poro que contiene al Ni se cuenta dos veces como anillo de 30, porque se puede rodear pasando por el B o por el Ni con el mismo número de pasos.
* **Con Ni–B**: aparecen dos triángulos Ni–B–C.

En estudios con metales suele ser más claro calcular la estadística **solo sobre la red B–C** (sin incluir el metal) y analizar aparte la coordinación del metal.

## 11. Cómo aplicarlo a una trayectoria de MD real

Para hacer lo mismo que el artículo (instantáneas de MD a cada temperatura, p. ej. con LAMMPS o con un MLIP), basta con leer los fotogramas y promediar. Plantilla:

In [ ]:
def ring_statistics_trajectory(frames, cutoff, max_size=60, criterion='guttman', exclude=()):
    # Media de N(n) por fotograma. 'exclude': elementos que no forman parte de la red (p. ej. ('Ni','H'))
    total, ncoord = Counter(), Counter()
    for at in frames:
        if exclude:
            at = at[[s not in exclude for s in at.get_chemical_symbols()]]
        r = ring_statistics(at, cutoff, max_size, criterion)
        total.update(r['counts']); ncoord.update(r['coordination'])
    nf = len(frames)
    return {n: c/nf for n, c in sorted(total.items())}, {z: c/nf for z, c in sorted(ncoord.items())}

# --- Ejemplo de uso con un archivo real (descomentar) ---
# from ase.io import read
# frames = read('md_1000K.extxyz', index='-50::5')                          # extxyz (ASE, MLIPs…)
# frames = read('dump.lammpstrj', index='::10', format='lammps-dump-text')   # LAMMPS
# for at in frames:                                                         # en LAMMPS los tipos son números:
#     at.set_chemical_symbols(['B' if t == 1 else 'C' for t in at.get_atomic_numbers()])
# N, Z = ring_statistics_trajectory(frames, cutoff=2.0, max_size=60)
# print(fractions(N))

# Demostración con "fotogramas" sintéticos:
frames = []
rng = np.random.default_rng(3)
for _ in range(4):
    x = sc.copy(); x.positions += rng.normal(0, 0.1, x.positions.shape); frames.append(x)
N, Z = ring_statistics_trajectory(frames, cutoff=2.0, max_size=60)
print('N(n) medio por fotograma:', N)
print('Z medio por fotograma    :', Z)

## 12. Lo mismo con `matscipy.rings.ring_statistics`

[`matscipy`](https://github.com/libAtoms/matscipy) es una librería de análisis para simulaciones atomísticas que incluye una implementación en C de la estadística de anillos:

```python
from matscipy.rings import ring_statistics
ringstat = ring_statistics(atoms, cutoff, maxlength=-1)
```

* `atoms`: objeto `ase.Atoms`.
* `cutoff`: un número o un diccionario por pares de elementos, p. ej. `{('B','C'): 2.0, ('C','C'): 2.0}`.
* `maxlength`: tamaño máximo de anillo que se busca (acelera el cálculo).
* Devuelve un array en el que la posición *n* es el número de anillos de tamaño *n*. Su longitud varía, así que lo convertimos a diccionario.

**Diferencias con el código propio:**

| | Código propio (secciones 5–11) | `matscipy` |
|---|---|---|
| Criterio | King, Guttman y SP estrictos | Solo **SP estrictos** (Franzblau, 1991) |
| Salida | Átomos de cada anillo (se pueden dibujar) | Solo recuentos por tamaño |
| Velocidad | Python puro | C: mucho más rápido que nuestro SP estricto, que es el criterio equivalente |
| Celdas pequeñas | Funciona (sigue las imágenes periódicas) | **Necesita una supercelda grande** (sección 12.2) |

In [ ]:
# Si falta:  %pip install matscipy
import time
import matscipy
from matscipy.rings import ring_statistics as ms_ring_statistics
print('matscipy', matscipy.__version__)

def ms_counts(atoms, cutoff, maxlength=-1):
    # Convierte la salida de matscipy (array indexado por tamaño) en {tamaño: número de anillos}
    arr = ms_ring_statistics(atoms, cutoff, maxlength)
    return {n: int(c) for n, c in enumerate(arr) if c}

### 12.1 Validación: grafeno y BGDY prístino

In [ ]:
gr4 = build_graphene().repeat((4, 4, 1))
print('Grafeno 4×4 | matscipy:', ms_counts(gr4, 1.7, 12), ' | código propio:', ring_statistics(gr4, 1.7, 12)['counts'])
print('BGDY 4×4    | matscipy:', ms_counts(sc, 2.0, 40), ' | código propio:', ring_statistics(sc, 2.0, 40)['counts'])

### 12.2 ⚠️ Cuidado con el tamaño de la supercelda

`matscipy` calcula las distancias en el grafo entre **índices de átomos**, sin distinguir imágenes periódicas. Por eso un anillo solo se detecta si "cabe" en la supercelda: si no, existe un atajo a través de una imagen periódica y el anillo se descarta.

Lo comprobamos aumentando la supercelda, con BGDY prístino y con la vacante de B de la sección 8 (poro de 60):

In [ ]:
print(f'{"supercelda":>10} | {"átomos":>6} | {"prístino: matscipy":>20} | {"prístino: propio":>17} | {"vacante: matscipy":>22}')
for n in [1, 2, 3, 4, 5, 6]:
    base = bgdy.repeat((n, n, 1))
    pr_ms = ms_counts(base, 2.0, 40)
    pr_own = ring_statistics(base, 2.0, 40)['counts']
    if n >= 3:
        v = base.copy(); del v[central_index(v, 'B')]
        vac_ms = ms_counts(v, 2.0, 80)
    else:
        vac_ms = '-'
    print(f'{n}×{n:<8} | {len(base):6d} | {str(pr_ms):>20} | {str(pr_own):>17} | {str(vac_ms):>22}')

* En la celda 1×1, `matscipy` no encuentra ningún anillo. El código propio sí encuentra el de 30, porque sigue los vectores de imagen.
* El poro de 60 de la vacante solo aparece a partir de una supercelda **5×5** (unos 59 Å de lado).

👉 **Regla práctica:** con `matscipy`, repite el cálculo con superceldas cada vez mayores hasta que los recuentos por celda no cambien. Para los anillos grandes que aparecen al desordenar BGDY, conviene trabajar con superceldas de al menos 6×6 u 8×8.

### 12.3 Vacante de B: `matscipy` frente a los tres criterios propios

Con una supercelda 6×6 ya suficiente:

In [ ]:
vac6 = bgdy.repeat((6, 6, 1)); del vac6[central_index(vac6, 'B')]
t0 = time.time(); r_ms = ms_counts(vac6, 2.0, 80); t_ms = time.time() - t0
r_g = ring_statistics(vac6, 2.0, 80, 'guttman')['counts']
r_k = ring_statistics(vac6, 2.0, 80, 'king')['counts']
t0 = time.time(); r_sp = dict(sorted(Counter(len(r) for r in find_sp_rings(build_graph(vac6, 2.0), 80)).items())); t_sp = time.time() - t0
print(f'matscipy (SP)       : {r_ms}   [{t_ms:.2f} s]')
print(f'propio SP estrictos : {r_sp}   [{t_sp:.2f} s]')
print(f'propio Guttman      : {r_g}')
print(f'propio King         : {r_k}')

`matscipy` coincide con nuestros anillos SP estrictos (33 poros de 30 y 1 poro de 60) y es mucho más rápido. Guttman y King se desvían por las razones explicadas en la sección 8.

### 12.4 Desorden "térmico" con `matscipy` (figura tipo Fig. 6)

Como `matscipy` es rápido, podemos usar una supercelda mayor (8×8, 896 átomos) y más configuraciones por σ. Es el mismo modelo de juguete de la sección 9.

In [ ]:
sc8 = bgdy.repeat((8, 8, 1))
ncells8 = len(sc8)//len(bgdy)

def ms_noisy(base, sigma, cutoff=2.0, nframes=10, maxlength=80, seed=0):
    rng = np.random.default_rng(seed)
    tot = Counter()
    for _ in range(nframes):
        x = base.copy(); x.positions += rng.normal(0, sigma, x.positions.shape)
        tot.update(ms_counts(x, cutoff, maxlength))
    return dict(sorted(tot.items()))

t0 = time.time()
ms_data = {s: ms_noisy(sc8, s) for s in sigmas}
print(f'Tiempo total ({len(sigmas)} σ × 10 configuraciones): {time.time()-t0:.1f} s')

fig, axes = plt.subplots(1, len(sigmas), figsize=(15, 3.2), sharey=True)
for ax, s in zip(axes, sigmas):
    f = fractions(ms_data[s])
    ax.bar(list(f.keys()), list(f.values()), color='#7F77DD', width=0.8)
    ax.set_title(f'σ = {s:.2f} Å'); ax.set_xlabel('Tamaño del anillo'); ax.set_xlim(0, 82)
axes[0].set_ylabel('Fracción de anillos')
plt.suptitle('matscipy · BGDY 8×8 · 10 configuraciones por σ', y=1.03)
plt.tight_layout(); plt.show()

for s in sigmas:
    print(f'σ = {s:.2f} Å  anillos de 30 por celda: {ms_data[s].get(30, 0)/(10*ncells8):.2f}   | todos: {ms_data[s]}')

Comprobación cruzada en una sola configuración desordenada (6×6, σ = 0.2 Å): `matscipy` y nuestros anillos SP estrictos deben dar exactamente lo mismo.

In [ ]:
x = bgdy.repeat((6, 6, 1)); x.positions += np.random.default_rng(0).normal(0, 0.2, x.positions.shape)
a = ms_counts(x, 2.0, 80)
b = dict(sorted(Counter(len(r) for r in find_sp_rings(build_graph(x, 2.0), 80)).items()))
c = ring_statistics(x, 2.0, 80, 'guttman')['counts']
print('matscipy      :', a)
print('propio SP     :', b, ' → iguales:', a == b)
print('propio Guttman:', c)

### 12.5 BGDY con Ni: radios de corte por pares

`matscipy` acepta el mismo diccionario de radios de corte por pares de elementos. Repetimos el ejemplo de la sección 10 (aquí en 3×3, porque los anillos son pequeños o de 30):

In [ ]:
print('Solo Ni–C   | matscipy:', ms_counts(ni, cut_A, 40), ' | propio SP:', dict(Counter(len(r) for r in find_sp_rings(build_graph(ni, cut_A), 40))))
print('Ni–C y Ni–B | matscipy:', ms_counts(ni, cut_B, 40), ' | propio SP:', dict(Counter(len(r) for r in find_sp_rings(build_graph(ni, cut_B), 40))))

### 12.6 Trayectorias de MD con `matscipy`

La misma plantilla de la sección 11, pero con `matscipy`. Es la opción recomendable para trayectorias largas y sistemas grandes.

In [ ]:
def ms_ring_statistics_trajectory(frames, cutoff, maxlength=80, exclude=()):
    # Media de N(n) por fotograma con matscipy. 'exclude': elementos fuera de la red (p. ej. ('Ni','H'))
    total = Counter()
    for at in frames:
        if exclude:
            at = at[[s not in exclude for s in at.get_chemical_symbols()]]
        total.update(ms_counts(at, cutoff, maxlength))
    return {n: c/len(frames) for n, c in sorted(total.items())}

# from ase.io import read
# frames = read('md_1000K.extxyz', index='::10')
# N = ms_ring_statistics_trajectory(frames, cutoff={('B','C'): 2.0, ('C','C'): 2.0}, maxlength=80)
# print(fractions(N))

frames8 = []
rng = np.random.default_rng(5)
for _ in range(5):
    y = sc8.copy(); y.positions += rng.normal(0, 0.12, y.positions.shape); frames8.append(y)
print('N(n) medio por fotograma (8×8, σ = 0.12 Å):', ms_ring_statistics_trajectory(frames8, 2.0, 80))

### 12.7 Rendimiento

Tiempo de una configuración desordenada (σ = 0.15 Å, `maxlength` = 80) según el tamaño del sistema:

In [ ]:
print(f'{"supercelda":>10} | {"átomos":>6} | {"matscipy SP (s)":>15} | {"propio SP (s)":>13} | {"propio Guttman (s)":>18}')
rng = np.random.default_rng(2)
for n in [4, 8, 12, 16]:
    x = bgdy.repeat((n, n, 1)); x.positions += rng.normal(0, 0.15, x.positions.shape)
    t0 = time.time(); ms_counts(x, 2.0, 80); tm = time.time() - t0
    if n <= 8:
        t0 = time.time(); find_sp_rings(build_graph(x, 2.0), 80); ts = f'{time.time() - t0:13.2f}'
    else:
        ts = f'{"(lento)":>13}'
    t0 = time.time(); ring_statistics(x, 2.0, 80, 'guttman'); tg = time.time() - t0
    print(f'{n}×{n:<8} | {len(x):6d} | {tm:15.2f} | {ts} | {tg:18.2f}')

* Con el mismo criterio (SP estrictos), `matscipy` es decenas de veces más rápido que nuestra implementación en Python (entre 30 y 70 veces en esta prueba).
* Nuestro Guttman tarda algo parecido a `matscipy`, pero es un criterio distinto: pierde algunos anillos grandes.
* En la versión 1.3.1, `ring_statistics` repite internamente la búsqueda de anillos dos veces, así que llamar directamente a `matscipy.rings.find_sp_rings` podría reducir el tiempo a la mitad.
* `matscipy` construye internamente una matriz N×N de distancias en el grafo. Para sistemas de decenas de miles de átomos, como los 20 064 del artículo, conviene vigilar la memoria o analizar regiones por separado.

## 13. Resumen y lista de comprobación

**Procedimiento (equivalente al del artículo):**
1. Tomar las configuraciones (instantáneas de MD) a cada temperatura.
2. Construir el grafo de enlaces con un radio de corte $r_c$ (primer mínimo de g(r)).
3. Encontrar los anillos con la regla del camino más corto (King o Guttman), respetando las condiciones periódicas.
4. Contar $N(n)$ y representar la fracción $f(n)$ frente al tamaño $n$ (Fig. 6), junto con la distribución de Z (Fig. 5).

**Resultados para BGDY:**
* Prístino: solo anillos de **30** (1 por celda unidad); Z = 3 para B y Z = 2 para C.
* Con defectos o desorden aparecen anillos de 50, 60, 70… (poros fusionados) y anillos pequeños (3, 29…) por acercamientos entre cadenas.

**Lo que conviene indicar siempre (el artículo no lo hace):**
- [ ] Radio de corte (y si es distinto por pares de elementos)
- [ ] Criterio de anillo: King, Guttman o SP/primitivos
- [ ] Tamaño máximo de anillo buscado (`max_size`)
- [ ] Normalización: fracción, anillos por celda o por átomo
- [ ] Número de instantáneas promediadas
- [ ] Si se incluyen o no átomos adsorbidos (metales, H) en la red

**¿Qué herramienta usar?**
* **`matscipy`**: rápido (C) y cómodo para sistemas grandes y trayectorias largas. Usa anillos SP estrictos (Franzblau), solo devuelve recuentos y necesita supercelda grande.
* **Código propio de este cuaderno**: más lento, pero da los átomos de cada anillo (para dibujarlos o analizarlos), funciona en celdas pequeñas e incluye King y Guttman, que es lo que usa ISAACS en el artículo.
* **ISAACS o R.I.N.G.S.**: si se quiere reproducir el artículo con la misma herramienta.